# Exploração · `bronze.inspecoes`

Notebook de **investigação**, fora do pipeline. Descobre os problemas de qualidade das inspeções de campo (planilha `inspecoes.xlsx`, preenchida à mão, com 3 abas unidas na Bronze) antes de escrever a Silver.

**Por que esta tabela importa:** é o histórico do que o time do Rafael encontrou em campo. Ela diz **quais inspeções acharam irregularidade**, e é isso que permite calibrar e validar a regra de prioridade da `fila_inspecao`. O Rafael diz que "pouco mais da metade" das inspeções encontra algo; a seção 11 mede isso e mostra o que diferencia as inspeções que acertaram.

Rode com **Run all**. Os achados vão nas células markdown e, depois, em `docs/data_quality_log.md` (seção `inspecoes`, códigos I).

| # | Seção | Pergunta |
|---|---|---|
| 1 | Visão geral | Colunas, volume por aba (o case fala em ~1,4 mil) |
| 2 | Nomes das colunas | Conferir os nomes reais antes de qualquer análise |
| 3 | Cabeçalho repetido | Há linha com o nome das colunas como dado? |
| 4 | Nulos e códigos de ausência | `N/A`, `NULL`, `-1`, `-`, vazio, `nan` |
| 5 | Duplicatas | Cópias exatas? A mesma inspeção em duas abas? |
| 6 | Formatos | Padrão de cada coluna |
| 7 | Data da inspeção | Formatos e ordem dia/mês, **com a aba como gabarito** |
| 8 | Resultado e tipo de irregularidade | Grafias e coerência entre os dois |
| 9 | Energia recuperada | Vírgula e ponto, valores impossíveis, coerência com o resultado |
| 10 | Integridade | Órfãs, UCs de teste, inspetores |
| 11 | O que acerta | Taxa de irregularidade por medidor, GD e queda de consumo (base da fila) |
| 12 | Resumo | Regras propostas para a Silver |

**Pré-requisito:** `silver.unidades_consumidoras`. A seção 11 usa `gold.dim_uc` se ela existir.

In [0]:
from pyspark.sql import functions as F

catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

ins_raw = spark.table(f"{bronze_schema}.inspecoes")

def n_br(n):
    return f"{n:,}".replace(",", ".")

def sem_acento(col):
    """upper + trim + sem acento + espaços simples: para comparar grafias."""
    c = F.translate(F.upper(F.trim(col)), "ÁÀÂÃÉÊÍÓÔÕÚÜÇ", "AAAAEEIOOOUUC")
    return F.regexp_replace(c, r"\s+", " ")

def padrao(col):
    """Troca dígitos por 9 e letras por A: mostra o formato do valor."""
    return F.regexp_replace(F.regexp_replace(col, "[0-9]", "9"), "[A-Za-zÀ-ÿ]", "A")

## 1. Visão geral

**Achado:** 1.389 linhas, como o case informa (~1,4 mil): `ago-set` 511, `out-nov` 495 e `dez` 383. Oito colunas de origem, todas texto.

In [0]:
ins_raw.printSchema()
print(f"{n_br(ins_raw.count())} linhas (o case fala em ~1,4 mil)")
display(ins_raw.groupBy("aba_origem").count().orderBy("aba_origem"))
display(ins_raw.limit(10))

## 2. Nomes das colunas

**Esta é a única célula que depende dos nomes das colunas.** Compare com o `printSchema` da seção 1 e **corrija aqui** o que estiver diferente. Se uma coluna não existir, troque por `None`.

A célula para com erro se um nome preenchido não existir.

In [0]:
COL_ID = "id_inspecao"
COL_UC = "id_uc"
COL_DATA = "data"
COL_RESULTADO = "resultado"
COL_TIPO = "tipo_irregularidade"   # None se não existir
COL_KWH = "kwh_recuperado"          # None se não existir
COL_INSPETOR = "inspetor"           # None se não existir
COL_ABA = "aba_origem"              # criada na Bronze

preenchidas = [c for c in [COL_ID, COL_UC, COL_DATA, COL_RESULTADO, COL_TIPO, COL_KWH, COL_INSPETOR, COL_ABA] if c]
faltando = [c for c in preenchidas if c not in ins_raw.columns]
if faltando:
    raise ValueError(f"Colunas não encontradas: {faltando}. Colunas reais: {ins_raw.columns}. Corrija os nomes nesta célula.")

cols_origem = [c for c in ins_raw.columns if c not in ("source_file", "ingestion_timestamp")]
print("Colunas confirmadas:", preenchidas)
print("Todas as colunas de origem:", cols_origem)

## 3. Cabeçalho repetido

**Achado:** nenhuma linha de cabeçalho repetido.

In [0]:
cab = ins_raw.filter(F.col(COL_ID) == COL_ID)
print("linhas de cabeçalho:", cab.count())
ins = ins_raw.filter(~F.coalesce(F.col(COL_ID) == COL_ID, F.lit(False)))
print("linhas sem cabeçalho:", n_br(ins.count()))

## 4. Nulos e códigos de ausência

**O que explora:** nulos reais e as formas de "vazio" das outras tabelas. Inclui `nan` e `None` como texto, porque o Excel foi lido com pandas na Bronze e células vazias podem ter virado esses textos.

**Achado:** só `tipo_irregularidade` tem ausência: nulo real (441) e `-` (77), total de **518**. Nenhum `nan` ou `None` como texto vindo do pandas.

In [0]:
codigos = ["N/A", "NULL", "-1", "-", "", "nan", "None", "9999-12-31"]
linhas = []
for c in cols_origem:
    linha = {"coluna": c, "nulo_real": ins.filter(F.col(c).isNull()).count()}
    for cod in codigos:
        linha[f"'{cod}'"] = ins.filter(F.trim(F.col(c)) == cod).count()
    linhas.append(linha)
display(spark.createDataFrame(linhas))

## 5. Duplicatas

**O que explora:** cópias exatas; o mesmo `id_inspecao` com conteúdo diferente (por exemplo, a mesma inspeção lançada em duas abas); e a mesma UC inspecionada mais de uma vez (pode ser legítimo: reinspeção).

**Achado:** sem cópias (1.389 linhas e 1.389 ids distintos) e nenhum id com versões diferentes. A mesma inspeção não aparece em duas abas.

In [0]:
print("linhas:                          ", n_br(ins.count()))
print("linhas distintas (sem auditoria e sem aba):",
      n_br(ins.select([c for c in cols_origem if c != COL_ABA]).distinct().count()))
print("id_inspecao distintos:           ", n_br(ins.select(COL_ID).distinct().count()))

versoes = ins.select(cols_origem).distinct()
ids_multi = versoes.groupBy(COL_ID).count().filter("count > 1").select(COL_ID)
print("ids com versões diferentes:", n_br(ids_multi.count()))
dif = versoes.join(ids_multi, COL_ID)
if ids_multi.count() > 0:
    display(dif.groupBy(COL_ID).agg(
        *[(F.countDistinct(F.coalesce(F.col(c), F.lit("<nulo>"))) > 1).cast("int").alias(c)
          for c in cols_origem if c != COL_ID]
    ).select([F.sum(c).alias(c) for c in cols_origem if c != COL_ID]))
    display(dif.orderBy(COL_ID).limit(20))

# A mesma UC inspecionada mais de uma vez
display(ins.dropDuplicates([COL_ID]).groupBy(COL_UC).count()
        .groupBy(F.col("count").alias("inspecoes_por_uc")).agg(F.count("*").alias("qtd_ucs")).orderBy("inspecoes_por_uc"))

## 6. Formatos

**O que explora:** o padrão de cada coluna (dígitos viram `9`, letras viram `A`).

**Achado:** ids, UC e inspetor com padrão único. Dois problemas de formato: **`data` em 4 formatos** (ISO 1.256; hífen 46; ponto 44; barra 43) e **`kwh_recuperado` com vírgula decimal em 125 linhas**.

In [0]:
for c in cols_origem:
    print(c)
    display(ins.groupBy(padrao(F.trim(F.col(c))).alias("padrao")).count().orderBy(F.desc("count")).limit(12))

## 7. Data da inspeção: formatos e ordem dia/mês

A Bronze já mostrou 3 formatos: ISO (`2025-08-01`), ponto (`14.08.2025`) e hífen (`09-01-2025`). A convenção do projeto é hífen = mês primeiro; ponto e barra = dia primeiro.

**Gabarito desta tabela:** cada aba do Excel corresponde a um período (`ago-set`, `out-nov`, `dez`). Com a ordem certa, o mês de **toda** inspeção cai dentro da aba de origem. Com a ordem trocada, não. É o mesmo tipo de teste do `mes_referencia` em leituras.

**Achado:**
- **Ordem dia/mês:** a convenção do projeto (hífen = mês primeiro; ponto e barra = dia primeiro) converte **100%** das datas. A ordem trocada deixa sem converter 33 de 46 (hífen), 27 de 44 (ponto) e 26 de 43 (barra). Só a convenção é possível.
- **Gabarito da aba:** as 1.256 datas ISO caem **todas** no período da própria aba. As **107 datas fora da aba estão todas na aba `dez`**, com meses de agosto a novembro (24, 23, 34 e 26), distribuídas de forma regular. Nenhuma ordem de leitura as coloca em dezembro. Interpretação: a aba `dez` recebeu também inspeções **feitas antes e lançadas em dezembro**. A aba indica o mês de lançamento, e não necessariamente o da inspeção.
- **Período:** todas as inspeções entre agosto e dezembro de 2025.

In [0]:
meses_aba = {"ago-set": [8, 9], "out-nov": [10, 11], "dez": [12]}  # confira os nomes das abas na seção 1
abas_reais = [r[COL_ABA] for r in ins.select(COL_ABA).distinct().collect()]
print("abas encontradas:", abas_reais, "| abas no gabarito:", list(meses_aba))

def converte(col, fmt):
    return F.expr(f"to_date(try_to_timestamp(trim({col}), '{fmt}'))")

dt = ins.withColumn("padrao_data", padrao(F.trim(F.col(COL_DATA))))
display(dt.groupBy("padrao_data").count().orderBy(F.desc("count")))

mes_ok = None
for aba, meses in meses_aba.items():
    c = (F.col(COL_ABA) == aba) & F.month("d").isin(meses)
    mes_ok = c if mes_ok is None else (mes_ok | c)

candidatos = {
    "9999-99-99": ["yyyy-MM-dd"],
    "99-99-9999": ["MM-dd-yyyy", "dd-MM-yyyy"],
    "99.99.9999": ["dd.MM.yyyy", "MM.dd.yyyy"],
    "99/99/9999": ["dd/MM/yyyy", "MM/dd/yyyy"],
}
for p, fmts in candidatos.items():
    sub = dt.filter(F.col("padrao_data") == p)
    n = sub.count()
    if n == 0:
        continue
    for fmt in fmts:
        conv = sub.withColumn("d", converte(COL_DATA, fmt))
        print(f"{p} como {fmt}: linhas {n_br(n)} | não converte {n_br(conv.filter('d IS NULL').count())}"
              f" | mês dentro da aba {n_br(conv.filter(mes_ok).count())}")

In [0]:
# Com a convenção do projeto: datas fora da aba, ano e período
formatos = {"9999-99-99": "yyyy-MM-dd", "99-99-9999": "MM-dd-yyyy", "99.99.9999": "dd.MM.yyyy", "99/99/9999": "dd/MM/yyyy"}
conv = None
for p, fmt in formatos.items():
    cond, val = F.col("padrao_data") == p, converte(COL_DATA, fmt)
    conv = F.when(cond, val) if conv is None else conv.when(cond, val)
ins_dt = dt.withColumn("d", conv)
print("não converteu:", n_br(ins_dt.filter("d IS NULL").count()))
print("mês fora da aba:", n_br(ins_dt.filter(F.col("d").isNotNull() & ~mes_ok).count()))
display(ins_dt.groupBy(F.year("d").alias("ano"), F.month("d").alias("mes"), COL_ABA).count().orderBy("ano", "mes"))

## 8. Resultado e tipo de irregularidade

**O que explora:** todas as grafias do resultado, para montar um **mapa explícito** (valor a valor), e não uma regra por expressão regular. Uma regra como "contém CONFIRMADA" marcaria `IRREGULARIDADE NAO CONFIRMADA` como confirmada, invertendo o sentido justamente da coluna que calibra a fila.

Também cruza o resultado com o tipo de irregularidade: se não houve irregularidade, o tipo deveria estar vazio.

**Achado:**
- **Resultado:** 8 grafias para 3 significados. `IRREGULARIDADE CONFIRMADA` (777) + variações de maiúsculas (32 + 27) + `CONFIRMADA` (35) = **871 confirmadas**. `SEM IRREGULARIDADE` (448) + variações (33 + 15) = 496, e **`NADA CONSTA` (22)**.
- **`NADA CONSTA` significa sem irregularidade:** nas 22, o tipo está vazio e a energia recuperada é zero, igual às `SEM IRREGULARIDADE`.
- **Coerência perfeita:** todo resultado confirmado tem tipo preenchido, e todo resultado sem irregularidade tem tipo vazio (as 518 ausências da seção 4 são exatamente esses casos).
- **Tipo de irregularidade:** 5 valores limpos: `DESVIO DIRETO` (182), `BY-PASS` (181), `MEDIDOR ADULTERADO` (180), `LACRE VIOLADO` (164) e `LIGACAO CLANDESTINA` (164).

In [0]:
print("Resultado: valores brutos e padronizados (sem acento, maiúsculas)")
display(ins.groupBy(F.col(COL_RESULTADO).alias("bruto"), sem_acento(F.col(COL_RESULTADO)).alias("padronizado"))
           .count().orderBy("padronizado", F.desc("count")))

if COL_TIPO:
    print("Tipo de irregularidade:")
    display(ins.groupBy(sem_acento(F.col(COL_TIPO)).alias("tipo")).count().orderBy(F.desc("count")))
    print("Resultado x tipo (o tipo deveria ser vazio quando não há irregularidade):")
    display(ins.groupBy(sem_acento(F.col(COL_RESULTADO)).alias("resultado"),
                        F.when(F.trim(F.col(COL_TIPO)).isin("-", "", "nan", "None") | F.col(COL_TIPO).isNull(), "VAZIO")
                         .otherwise("PREENCHIDO").alias("tipo"))
               .count().orderBy("resultado"))

## 9. Energia recuperada

**O que explora:** vírgula e ponto decimal na mesma coluna (`1313,70` e `838.10`), valores que não convertem, zero ou negativos, e a coerência com o resultado: só deveria haver energia recuperada quando a irregularidade foi confirmada.

**Achado:** 125 valores com vírgula decimal, todos convertem. **Energia recuperada positiva em 100% das 871 confirmadas** (mediana de ~1.072 kWh) e **zero em 100% das 518 sem irregularidade**. Máximo de 15.609 kWh; percentil 99 de 8.897 kWh. Nenhum valor negativo.

In [0]:
if COL_KWH:
    kwh = ins.withColumn("kwh", F.expr(f"try_cast(regexp_replace(trim({COL_KWH}), ',', '.') as double)"))
    print("com vírgula decimal:", n_br(kwh.filter(F.col(COL_KWH).contains(",")).count()))
    print("não converte:", n_br(kwh.filter(F.col(COL_KWH).isNotNull() & F.col("kwh").isNull()
                                         & ~F.trim(F.col(COL_KWH)).isin("-", "", "nan", "None")).count()))
    print("zero:", n_br(kwh.filter("kwh = 0").count()), "| negativo:", n_br(kwh.filter("kwh < 0").count()))
    display(kwh.select(F.min("kwh"), F.expr("percentile_approx(kwh, array(0.25, 0.5, 0.75, 0.99))").alias("p25_mediana_p75_p99"), F.max("kwh")))
    print("Energia recuperada por resultado:")
    display(kwh.groupBy(sem_acento(F.col(COL_RESULTADO)).alias("resultado")).agg(
        F.count("*").alias("inspecoes"), F.sum(F.col("kwh").isNotNull().cast("int")).alias("com_kwh"),
        F.sum((F.col("kwh") > 0).cast("int")).alias("kwh_positivo"), F.round(F.expr("percentile_approx(kwh, 0.5)"), 1).alias("mediana_kwh")))

## 10. Integridade

**O que explora:** `id_uc` no padrão e existente na `silver.unidades_consumidoras`; inspeções em UCs de teste; padrão do inspetor (`INSP-000`) e quantas inspeções cada um fez.

**Achado:** nenhuma órfã, nenhuma inspeção em UC de teste. **40 inspetores**, todos no padrão `INSP-000` depois do `trim`, com 24 a 48 inspeções cada.

In [0]:
ucs = spark.table(f"{silver_schema}.unidades_consumidoras").select("id_uc")
ucs_bronze = (spark.table(f"{bronze_schema}.unidades_consumidoras")
              .filter(F.col("id_uc") != "id_uc").select("id_uc").distinct())
ucs_teste = ucs_bronze.join(ucs, "id_uc", "left_anti")

i = ins.dropDuplicates([COL_ID]).withColumn("id_uc", F.upper(F.trim(F.col(COL_UC))))
print("id_uc fora do padrão UC + 6 dígitos:",
      n_br(i.filter(~F.coalesce(F.col("id_uc").rlike("^UC[0-9]{6}$"), F.lit(False))).count()))
print("id_uc inexistente na Silver de UCs:  ", n_br(i.join(ucs, "id_uc", "left_anti").count()))
print("inspeções em UCs de teste:           ", n_br(i.join(ucs_teste, "id_uc", "left_semi").count()))
display(i.join(ucs, "id_uc", "left_anti").groupBy(padrao(F.col("id_uc")).alias("padrao")).count())

if COL_INSPETOR:
    insp = i.withColumn("insp", F.upper(F.trim(F.col(COL_INSPETOR))))
    print("inspetor fora do padrão INSP-000:", n_br(insp.filter(~F.coalesce(F.col("insp").rlike("^INSP-[0-9]{3}$"), F.lit(False))).count()))
    display(insp.groupBy("insp").count().orderBy(F.desc("count")))

## 11. O que acerta: base para a regra da fila de perdas

**O que explora:** a taxa de inspeções com irregularidade confirmada, no total e por característica da UC. Se alguma característica tem taxa bem maior que a média, ela merece peso na regra do `score` da `fila_inspecao`.

**Atenção:** antes de rodar, ajuste a lista `CONFIRMADA` abaixo com os valores reais da seção 8 que significam "irregularidade encontrada".

**Achado:** **taxa geral de 62,7%** (871 de 1.389), compatível com o "pouco mais da metade" do Rafael. A taxa por característica da UC (medidor, GD, queda de consumo) fica para depois da `gold.dim_uc`: é a base da regra de prioridade da `fila_inspecao`.

In [0]:
# Ajuste com os valores padronizados da seção 8 que significam irregularidade encontrada
CONFIRMADA = ["IRREGULARIDADE CONFIRMADA", "CONFIRMADA"]  # confirmado na seção 8

base = i.withColumn("confirmada", sem_acento(F.col(COL_RESULTADO)).isin(CONFIRMADA))
tot = base.agg(F.count("*").alias("inspecoes"), F.sum(F.col("confirmada").cast("int")).alias("confirmadas")).first()
print(f"Taxa geral: {tot['confirmadas']} de {tot['inspecoes']} = {tot['confirmadas'] / tot['inspecoes']:.1%} (o Rafael fala em 'pouco mais da metade')")

def taxa(df, col):
    display(df.groupBy(col).agg(F.count("*").alias("inspecoes"),
                                F.round(F.avg(F.col("confirmada").cast("int")) * 100, 1).alias("taxa_confirmada_pct"))
              .orderBy(F.desc("taxa_confirmada_pct")))

if spark.catalog.tableExists(f"{gold_schema}.dim_uc"):
    d = spark.table(f"{gold_schema}.dim_uc")
    b = base.join(d, "id_uc", "left")
    taxa(b, "classe")
    taxa(b, "tipo_medidor_ativo")
    taxa(b.withColumn("faixa_idade_medidor", F.when(F.col("idade_medidor_anos") < 5, "0-5").when(F.col("idade_medidor_anos") < 10, "5-10")
                                               .when(F.col("idade_medidor_anos") < 20, "10-20").otherwise("20+")), "faixa_idade_medidor")
    taxa(b, "possui_gd")
    taxa(b, "medidor_trocado_recentemente")
    taxa(b.withColumn("faixa_variacao_consumo", F.when(F.col("variacao_consumo_12m_pct") < -30, "queda > 30%")
                                                  .when(F.col("variacao_consumo_12m_pct") < -10, "queda 10-30%")
                                                  .when(F.col("variacao_consumo_12m_pct") < 10, "estável")
                                                  .when(F.col("variacao_consumo_12m_pct").isNotNull(), "alta > 10%")
                                                  .otherwise("sem histórico")), "faixa_variacao_consumo")
else:
    print("gold.dim_uc ainda não existe: rode esta seção depois de gravar a dim_uc.")

## 12. Resumo e regras para a Silver

| Problema | Linhas | Regra | Log |
|---|---|---|---|
| `data` em 4 formatos | 133 fora do ISO | cada padrão com o seu formato (hífen = mês primeiro) | I1 |
| Data fora do período da aba | 107 (todas na aba `dez`) | manter a data + flag `data_fora_da_aba` | I2 |
| `resultado` com 8 grafias | 164 fora dos 2 valores padrão | mapa explícito para `IRREGULARIDADE CONFIRMADA` e `SEM IRREGULARIDADE` (`NADA CONSTA` → sem irregularidade) + flag `irregularidade_confirmada` | I3 |
| `tipo_irregularidade` ausente em 2 formas | 518 | nulo real (`-` vira nulo) | I4 |
| `kwh_recuperado` com vírgula | 125 | vírgula vira ponto, `decimal(12,2)` | I5 |

**Verificado e sem problema:** sem cabeçalho, cópias, órfãs ou UCs de teste; 40 inspetores no padrão; tipo preenchido se e somente se a irregularidade foi confirmada; energia recuperada positiva se e somente se confirmada.

**Para a fila de perdas:** taxa de acerto de 62,7%. As taxas por característica saem da seção 11, depois da `dim_uc`.